# 25. Real photos: five-column check

Runs the trained models and the ensemble on the real photos in `data/real` and writes one picture per photo with five columns:

1. original
2. background subtraction (classical)
3. PatchCore (trained on AITEX good tiles)
4. PaDiM (tuned settings from notebook 23)
5. ensemble `max` = pixel-wise maximum of the z-scored background-subtraction and PatchCore maps. Red boxes = 256 x 256 tiles the ensemble flags as DAMAGED.

PaDiM is shown for comparison only; it is **not** a member of the ensemble (members are the same as in notebook 24: background subtraction + PatchCore).

**Read the result with care.** There are no photos of undamaged real fabric, so the OK / DAMAGED decision uses the thresholds of AITEX fabric 02. A bright area only means "most different inside this photo": chalk, stickers, labels, edges and the table also light up. Add good photos of the real fabric to `data/real_good` to make a valid threshold (see `USE_REAL_GOOD` below).

The ensemble heat map and the tile decision are two different things: the heat map is for looking, the red boxes come from `E.ensemble_judge` (same rule as notebook 24).

In [1]:
import sys, glob, os, time, importlib
import numpy as np, cv2, torch
import matplotlib.pyplot as plt

sys.path.insert(0, "../src/methods")
import filters as F
import evaluate as E
import models as M
importlib.reload(F); importlib.reload(E); importlib.reload(M)
print(M.device_info())

# ---- settings ---------------------------------------------------------------
PHOTO_DIR = "../data/real"            # the real photos (jpg / jpeg / png)
OUT_DIR = "../results/real_photos"    # one five-column picture per photo
TRAIN_FABRIC = "02"                   # AITEX fabric whose good strips train the models and set the thresholds
FIT_FRACTION = 0.7                    # share of good_cal strips that trains the models, the rest sets thresholds
RULE = "max"                          # ensemble rule: "max" | "mean" | "min" (same as notebook 24)
LONG_SIDE = 640                       # photos are resized so the long side has this many px
WEIGHTS = M.WEIGHTS                   # backbone weights (name or path to a saved state_dict)
SHOW = 3                              # how many photos to draw in this notebook (all are saved to OUT_DIR)

# PaDiM (tuned) settings = PARAMS of notebook 23
PADIM = dict(backbone="wide_resnet50_2", layers=("layer2", "layer3"), n_dims=300)

# optional: photos of UNDAMAGED real fabric. When there are some, the ensemble is calibrated on them
# instead of AITEX (see the last cells). Leave the folder empty / missing to use AITEX fabric 02.
REAL_GOOD_DIR = "../data/real_good"
torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))
os.makedirs(OUT_DIR, exist_ok=True)
files = sorted(f for ext in ("jpg", "jpeg", "png", "JPG", "JPEG", "PNG") for f in glob.glob(os.path.join(PHOTO_DIR, "*." + ext)))
print(len(files), "photos in", PHOTO_DIR)

torch 2.15.0+cpu, device: cpu (no GPU found: slower, still works)
46 photos in ../data/real


## 1. Fit the models on AITEX good tiles
Only good strips of one fabric are used, the same strips as in the notebooks (`good_cal`, first `FIT_FRACTION` of them). Takes a few minutes on CPU.

In [3]:
good, _, _, _ = E.collect(TRAIN_FABRIC)
cal, _ = E.split_by_source(good)
fit_paths, _ = E.split_by_source(cal, FIT_FRACTION, 1)
fit_tiles = [E.load(p) for p in fit_paths]
print(len(fit_tiles), "good tiles used to fit")

t0 = time.time()
pc = M.fit_patchcore(fit_tiles, weights=WEIGHTS)
pd_ = M.fit_padim(fit_tiles, weights=WEIGHTS, **PADIM)
print(f"fitted in {time.time() - t0:.0f} s")

# the ensemble decision per tile, calibrated exactly as in notebook 24
MEMBERS = {
    "background_subtraction": dict(method=lambda t: F.background_subtraction(t)),
    "patchcore": dict(fit=lambda ts: (lambda f: (lambda t: M.patchcore_map(t, f)))(M.fit_patchcore(ts, weights=WEIGHTS))),
}
ens = E.calibrate_ensemble(MEMBERS, TRAIN_FABRIC, RULE, fit_fraction=FIT_FRACTION)
print("ensemble calibrated on AITEX fabric", TRAIN_FABRIC)

98 good tiles used to fit


RuntimeError: Pivots given to lu_solve must all be greater or equal to 1. Did you properly pass the result of lu_factor?

## 2. Helpers
`tiled` runs a 256 x 256 tile method (PaDiM) over a whole photo with overlapping tiles and averages the overlaps. `z` puts a map on the "robust deviations from its own median" scale so two maps can be combined with `max`.

In [ ]:
def load_photo(path, long_side=LONG_SIDE):
    im = cv2.imread(path)
    h, w = im.shape[:2]
    s = long_side / max(h, w)
    im = cv2.resize(im, (int(round(w * s / 8)) * 8, int(round(h * s / 8)) * 8), interpolation=cv2.INTER_AREA)
    return im, cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)

def starts(n, t=256):
    if n <= t:
        return [0]
    k = int(np.ceil((n - t) / (t * 0.75))) + 1
    return [int(round(i * (n - t) / (k - 1))) for i in range(k)]

def tiled(gray, fn, t=256):
    acc = np.zeros(gray.shape, np.float32); cnt = np.zeros(gray.shape, np.float32)
    for y in starts(gray.shape[0]):
        for x in starts(gray.shape[1]):
            acc[y:y + t, x:x + t] += fn(gray[y:y + t, x:x + t]); cnt[y:y + t, x:x + t] += 1
    return acc / cnt

def z(m, sigma=2.0):
    m = cv2.GaussianBlur(m, (0, 0), sigma)
    med = np.median(m); mad = 1.4826 * np.median(np.abs(m - med)) + 1e-6
    return (m - med) / mad

def label(img, text):
    cv2.rectangle(img, (0, 0), (img.shape[1], 22), (0, 0, 0), -1)
    cv2.putText(img, text, (6, 16), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
    return img

def heat(gray, amap, text):
    lo, hi = np.percentile(amap, 50), np.percentile(amap, 99.9)
    u = np.uint8(np.clip((amap - lo) / (hi - lo + 1e-6), 0, 1) * 255)
    o = cv2.addWeighted(cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR), 0.45, cv2.applyColorMap(u, cv2.COLORMAP_INFERNO), 0.55, 0)
    return label(o, text)

def five_columns(path, idx):
    im, g = load_photo(path)
    bg = F.background_subtraction(g)          # whole photo
    pcm = M.patchcore_map(g, pc)              # whole photo
    pdm = tiled(g, lambda t: M.padim_map(t, pd_))
    ensm = np.maximum(z(bg), z(pcm))          # ensemble heat map (max)
    n = k = kb = kp = 0; boxes = []
    for y in starts(g.shape[0]):
        for x in starts(g.shape[1]):
            s, flagged, det = E.ensemble_judge(g[y:y + 256, x:x + 256], ens)
            n += 1; k += int(flagged); kb += int(det[0]["flagged"]); kp += int(det[1]["flagged"])
            if flagged:
                boxes.append((x, y))
    cols = [label(im.copy(), f"{idx:02d}  original"),
            heat(g, bg, "background subtraction"),
            heat(g, pcm, "PatchCore (trained on AITEX)"),
            heat(g, pdm, "PaDiM (tuned)"),
            heat(g, ensm, f"ensemble {RULE}: {k} of {n} tiles flagged")]
    for x, y in boxes:
        cv2.rectangle(cols[-1], (x + 3, max(y + 3, 25)), (x + 252, y + 252), (0, 0, 255), 1)
    cols = [cv2.copyMakeBorder(c, 0, 0, 0, 6, cv2.BORDER_CONSTANT, value=(255, 255, 255)) for c in cols]
    return np.hstack(cols), (n, k, kb, kp)

## 3. Run all photos
Each picture is saved to `OUT_DIR` (`photo00.jpg`, ...) and the first `SHOW` are drawn here. `index.txt` lists which file is which photo.

In [ ]:
stats = []
with open(os.path.join(OUT_DIR, "index.txt"), "w") as fh:
    for i, f in enumerate(files):
        t0 = time.time()
        pic, st = five_columns(f, i)
        cv2.imwrite(os.path.join(OUT_DIR, f"photo{i:02d}.jpg"), pic, [cv2.IMWRITE_JPEG_QUALITY, 82])
        stats.append(st); fh.write(f"photo{i:02d}.jpg\t{os.path.basename(f)}\n")
        print(f"photo{i:02d}: tiles {st[0]}, ensemble flags {st[1]}, bg-sub {st[2]}, patchcore {st[3]}  ({time.time() - t0:.0f} s)")
        if i < SHOW:
            plt.figure(figsize=(22, 4.5)); plt.imshow(cv2.cvtColor(pic, cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.show()

a = np.array(stats)
print("\nTOTAL tiles", a[:, 0].sum(), "| ensemble flagged", a[:, 1].sum(), f"({a[:, 1].sum() / a[:, 0].sum():.0%})",
      "| background subtraction", a[:, 2].sum(), "| PatchCore", a[:, 3].sum(),
      "| photos with every tile flagged", int((a[:, 0] == a[:, 1]).sum()), "| photos with no flag", int((a[:, 1] == 0).sum()))

## 4. How to read this
- Every photo here shows damage, so "flagged" is not a false alarm or a hit you can count: there are no good photos to measure false alarms on.
- If almost every tile is flagged, the threshold from AITEX does not fit these photos (different fabric, colour, lighting, hand-held). That is a calibration problem, not proof that every tile is damaged.
- Next step for a valid result: put photos of **undamaged** real fabric into `data/real_good`, then run the cell below. It trains PatchCore on those photos and sets the threshold from them.

In [ ]:
good_files = sorted(f for ext in ("jpg", "jpeg", "png", "JPG", "JPEG", "PNG") for f in glob.glob(os.path.join(REAL_GOOD_DIR, "*." + ext)))
if not good_files:
    print(f"no photos in {REAL_GOOD_DIR}: add photos of undamaged fabric to calibrate on the real fabric")
else:
    # cut every good photo into overlapping 256 tiles; first FIT_FRACTION of the PHOTOS fit, the rest set the threshold
    def tiles_of(path):
        _, g = load_photo(path)
        return [g[y:y + 256, x:x + 256] for y in starts(g.shape[0]) for x in starts(g.shape[1]) if g[y:y + 256, x:x + 256].shape == (256, 256)]
    n_fit = max(1, int(round(len(good_files) * FIT_FRACTION)))
    fit_t = [t for f in good_files[:n_fit] for t in tiles_of(f)]
    cal_t = [t for f in good_files[n_fit:] for t in tiles_of(f)]
    print(len(good_files), "good photos ->", len(fit_t), "fit tiles,", len(cal_t), "calibration tiles")
    real_pc = M.fit_patchcore(fit_t, weights=WEIGHTS)
    cal_scores = np.array([F.tile_score(M.patchcore_map(t, real_pc), normalize=True) for t in cal_t])
    thr = F.robust_threshold(cal_scores)
    print(f"real-fabric PatchCore threshold = {thr:.3f}  (good tiles: median {np.median(cal_scores):.3f}, max {cal_scores.max():.3f})")
    flagged = []
    for i, f in enumerate(files):
        _, g = load_photo(f)
        sc = [F.tile_score(M.patchcore_map(g[y:y + 256, x:x + 256], real_pc), normalize=True)
              for y in starts(g.shape[0]) for x in starts(g.shape[1])]
        flagged.append((len(sc), int(np.sum(np.array(sc) > thr))))
        print(f"photo{i:02d}: {flagged[-1][1]} of {flagged[-1][0]} tiles flagged by PatchCore trained on real good fabric")